<a href="https://colab.research.google.com/github/mandaw34/BigData26_A_2411533009_Devina-Amanda-P/blob/main/Praktikum4/BD_P04_2411533009_Devina_Amanda_Putri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**K-1. Menyiapkan Lingkungan dan Data Contoh**

Menyiapkan pustaka penting seperti duckdb, networkx, shapely, tinydb, dan deltalake, serta memuat dataset trips_bersih dari praktikum sebelumnya. Selain melakukan standarisasi penamaan kolom TLC ke dalam format yang lebih intuitif (seperti waktu_jemput), sel ini juga mengambil sampel data sebanyak 300.000 baris agar proses pengujian model key-value dan dokumen berikutnya dapat berjalan efisien. Output dari sel ini berhasil menginisialisasi seluruh pustaka, memetakan ulang nama kolom, dan menghasilkan subset data bersih yang siap digunakan untuk eksperimen pemodelan selanjutnya.

In [1]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

from google.colab import drive
drive.mount("/content/drive")

import os, zipfile, time, json, sqlite3, random, shutil, glob
import numpy as np, pandas as pd
import duckdb

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
ZIP_TRIPS = f"{DIR_P3}/trips_bersih.zip"
TRIPS_BERSIH = "/content/trips_bersih"

os.makedirs(DIR_SIMPAN, exist_ok=True)

# Ekstrak file trips_bersih.zip jika belum diekstrak
if os.path.exists(ZIP_TRIPS) and not os.path.exists(f"{TRIPS_BERSIH}/extracted.flag"):
    print("Mengekstrak trips_bersih.zip...")
    os.makedirs(TRIPS_BERSIH, exist_ok=True)
    with zipfile.ZipFile(ZIP_TRIPS, 'r') as zip_ref:
        zip_ref.extractall(TRIPS_BERSIH)
    with open(f"{TRIPS_BERSIH}/extracted.flag", "w") as f:
        f.write("ok")

path_parquet = TRIPS_BERSIH
if os.path.exists(f"{TRIPS_BERSIH}/trips_bersih"):
    path_parquet = f"{TRIPS_BERSIH}/trips_bersih"

# Pengukur waktu (dari Praktikum 1/K-3)
catatan = []
def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil

# Membaca parquet menggunakan DuckDB untuk menghindari error ArrowInvalid dari PyArrow
def baca_parquet_duckdb():
    query = f"SELECT * FROM read_parquet('{path_parquet}/**/*.parquet', hive_partitioning=true)"
    return duckdb.query(query).df()

df = ukur("baca trips_bersih", baca_parquet_duckdb)

# Ganti nama kolom asli TLC menjadi nama yang lebih mudah dibaca
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}

cols_to_rename = {k: v for k, v in NAMA_BARU.items() if k in df.columns}
df = df.rename(columns=cols_to_rename)

if "wilayah_jemput" in df.columns:
    df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)

print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])

# Contoh 300.000 baris untuk K-2 dan K-3
sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
if "waktu_jemput" in sample.columns:
    sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[baca trips_bersih] 6.6126 s
baris: 2,986,910 | kolom: 18


**K-2. Tabel Biasa vs Key-value**

Melakukan eksperimen dan membandingkan performa antara tabel relasional konvensional berbasis SQLite dengan model key-value (menggunakan dictionary Python berisi format teks JSON) untuk pencarian data tunggal dan agregasi. Berdasarkan output eksekusi, pencarian data tunggal berdasarkan ID pada tabel relasional awalnya lambat namun melonjak sangat cepat setelah indeks dibuat hingga mampu menyaingi kecepatan model key-value, sementara model key-value menunjukkan kinerja agregasi massal (GROUP BY) yang jauh lebih lambat karena harus memparsing teks JSON satu per satu

In [2]:
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()

# Model 1 relasional: tabel SQLite
if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

# Model 2 key-value: kamus (dict) dengan kunci "perjalanan: <id>", nilainya teks JSON
def bangun_kv():
    global kv
    kv = {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}
    return kv

ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

# Pertanyaan A: ambil SATU perjalanan berdasarkan id (200 id acak)
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji]

def cari_kv():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan - SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan (id_perjalanan)") # buat indeks
r2 = ukur("cari 200 perjalanan - SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan - key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

# Pertanyaan B: hitung rata-rata total bayar per wilayah (melibatkan SEMUA data)
def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall()

def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks)
        w = d["wilayah_jemput"] # setiap nilai harus dibuka dulu
        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                  key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah - SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah - key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)

[relasional: muat tabel] 3.2904 s
[key-value: bangun dict] 9.7274 s
jumlah key: 300000
[cari 200 perjalanan - SQL tanpa indeks] 5.4908 s
[cari 200 perjalanan - SQL dengan indeks] 0.0038 s
[cari 200 perjalanan - key-value] 0.0012 s
hasil sama? True
[hitung per wilayah - SQL GROUP BY] 0.2024 s
[hitung per wilayah - key-value (buka semua nilai)] 1.2425 s
[('Manhattan', 267034, 22.7), ('Queens', 27288, 70.34), ('Unknown', 3787, 39.13)]
hasil sama? True


**K-3. Document: Bentuk Data yang Fleksibel**

Mendemonstrasikan implementasi model dokumen menggunakan TinyDB di mana data perjalanan direpresentasikan dalam bentuk struktur JSON bersarang (nested JSON). Output dari kode ini membuktikan bahwa model dokumen menawarkan fleksibilitas skema yang sangat tinggi, di mana atribut opsional (seperti kolom tip pada metode pembayaran tertentu) atau atribut baru (seperti data cuaca) dapat ditambahkan secara dinamis ke dalam dokumen tanpa memerlukan migrasi skema basis data formal.

In [3]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya": {
            "jarak_mil": float(r.jarak_mil),
            "total": float(r.total_bayar),
            "metode_bayar": r.metode_bayar
        },
    }
    # Tip hanya dicatat untuk pembayaran kartu -> field ini ada di sebagian dokumen saja
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)
print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

Q = Query()
print("jemput di Manhattan & total > 50:", len(tabel.search((Q.jemput.wilayah == "Manhattan") & (Q.biaya.total > 50))))
print("dokumen yang punya field tip:", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip:", len(tabel.search(~Q.biaya.tip.exists())))

# Menambah field baru ("cuaca") pada satu dokumen tanpa mengubah struktur apa pun
tabel.insert({
    "id_perjalanan": 999_999,
    "waktu_jemput": "2023-01-15 08:00:00",
    "jemput": {"zona": 161, "wilayah": "Manhattan"},
    "tujuan": {"zona": 236},
    "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip": 3.0},
    "cuaca": {"hujan": True, "suhu_c": -2.0}
})
print("dokumen yang punya field cuaca:", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang:", len(tabel))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-16 10:40:39",
  "jemput": {
    "zona": 48,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 164
  },
  "biaya": {
    "jarak_mil": 1.55,
    "total": 19.32,
    "metode_bayar": "Kartu kredit",
    "tip": 3.22
  }
}
jemput di Manhattan & total > 50: 693
dokumen yang punya field tip: 15906
dokumen yang TIDAK punya tip: 4094
dokumen yang punya field cuaca: 1
total dokumen sekarang: 20001


**K-4. Penyimpanan per Kolom: DuckDB pada Parquet**

Cell ini memanfaatkan mesin analitik DuckDB untuk mengeksekusi kueri SQL analitik secara langsung di atas berkas berorientasi kolom berformat Apache Parquet tanpa melalui proses impor tabel. Output menunjukkan performa kueri yang sangat cepat dan efisien, yang didukung oleh mekanisme columnar pruning, eksekusi paralel multi-inti, serta partition pruning berdasarkan wilayah, sehingga jauh lebih unggul dibandingkan basis data berbasis baris tradisional.

In [4]:
import duckdb
GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
con_d = duckdb.connect()

# VIEW = "jendela" bernama yang membaca Parquet langsung (tidak menyalin data).
con_d.sql(f"""
CREATE VIEW perjalanan AS
SELECT tpep_pickup_datetime AS waktu_jemput,
       PULocationID AS zona_jemput,
       DOLocationID AS zona_tujuan,
       borough_naik AS wilayah_jemput,
       trip_distance AS jarak_mil,
       total_amount AS total_bayar,
       nama_pembayaran AS metode_bayar
FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
         FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""
hasil_d = ukur("DuckDB hitung per wilayah (langsung dari Parquet)", lambda: con_d.sql(SQL).df())
print(hasil_d)

# Pembanding: SQLite (menyimpan per baris) pada data yang sama
KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
    os.remove("baris.db")
con_s = sqlite3.connect("baris.db")
ukur("SQLite muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s, index=False))
hasil_s = ukur("SQLite hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())

print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in hasil_s])

# Filter pada kolom partisi: DuckDB hanya membuka folder wilayah yang diminta
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
          FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))

ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, ds, fs in os.walk(TRIPS_BERSIH) for f in fs) / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {os.path.getsize('baris.db') / 1024**2:.1f} MB")

[DuckDB hitung per wilayah (langsung dari Parquet)] 0.1537 s
               wilayah_jemput   jumlah  rata_bayar
0                   Manhattan  2659609       22.72
1                      Queens   270315       70.46
2                     Unknown    37609       39.24
3                    Brooklyn    15724       32.72
4                       Bronx     3074       34.44
5  __HIVE_DEFAULT_PARTITION__      340       74.72
6               Staten Island      211       74.07
7                         EWR       28      106.41
[SQLite muat seluruh baris ke tabel] 15.1160 s
[SQLite hitung per wilayah] 1.8629 s
hasil sama? True
[DuckDB hanya wilayah Staten Island] 0.0159 s
[(211, 74.07)]
Parquet (18 kolom): 66.5 MB | SQLite (5 kolom): 114.2 MB


**K-5. Graph: Pertanyaan tentang Keterhubungan**

Pada tahap ini, data perjalanan diringkas ke dalam struktur jaringan (graph) menggunakan pustaka NetworkX, di mana zona mewakili simpul (node) dan arus perjalanan antar-zona merepresentasikan edge berbobot. Output dari kode ini membuktikan keunggulan model graph dalam menjawab pertanyaan relasional yang kompleks—seperti penelusuran jalur keterhubungan multi-langkah—secara lebih natural dan efisien tanpa memerlukan operasi self-join SQL yang rumit.

In [5]:
import networkx as nx

# Satu baris = satu arus: berapa perjalanan dari zona jemput ke zona tujuan tertentu
arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
    SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
    FROM perjalanan GROUP BY 1, 2""").df())

# Zona node (titik), arus edge (garis berarah, bobotnya jumlah perjalanan)
G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))
print("node (zona):", G.number_of_nodes(), " edge (arus):", G.number_of_edges())

# Lima arus terbesar
top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])
top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL", list(top_sql.itertuples(index=False, name=None)))

# Lima zona yang paling banyak menjadi tujuan
populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler:", populer)

MIN = 50
zona_awal = populer[0][0]

# (1) cara graph: telusuri tetangga dari tetangga
def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN:
            continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

# (2) cara SQL: self-join
con_d.register("arus", arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
          AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])

hg = ukur("2 langkah - graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah - SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal:", zona_awal, " terjangkau (graph):", len(hg), "| (SQL):", len(hs), " sama?", hg == hs)

# (3) Jumlah langkah bebas
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    print(f"dalam <= {k} langkah: {n} zona")

[hitung arus antar zona] 0.2141 s
node (zona): 262  edge (arus): 21583
graph: [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
SQL [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
zona tujuan terpopuler: [(236, 144209), (237, 130369), (161, 113991), (230, 87706), (170, 87110)]
[2 langkah - graph (NetworkX)] 0.0066 s
[2 langkah - SQL self-join (DuckDB)] 0.0067 s
zona awal: 236  terjangkau (graph): 214 | (SQL): 214  sama? True
dalam <= 1 langkah: 74 zona
dalam <= 2 langkah: 214 zona
dalam <= 3 langkah: 214 zona
dalam <= 4 langkah: 214 zona


**K-6. Indeks Spasial**

Mengevaluasi dan membandingkan efisiensi pencarian titik koordinat spasial di dalam wilayah tertentu menggunakan tiga metode berbeda: perulangan iteratif (loop), pemrosesan vektor NumPy, dan indeks spasial hierarkis berbasis STRtree. Output pengujian mengonfirmasi bahwa penggunaan indeks spasial mampu memangkas area pencarian yang tidak bersinggungan secara instan dan memberikan lonjakan performa yang signifikan dibandingkan metode pencarian konvensional pada volume data yang besar.

In [6]:
from shapely import STRtree, box, points

rng = np.random.default_rng(42)
N = 200_000
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)
bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70)
titik = points(bujur, lintang)

M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

# Cara 1: loop Python (hanya 5 kotak)
def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

# Cara 2: NumPy (tanpa indeks spasial)
def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        m = (bujur >= x - 0.005) & (bujur <= x + 0.005) & (lintang >= y - 0.005) & (lintang <= y + 0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop = ukur("cara 1 - loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 - NumPy, 1000 kotak", cara_numpy)

# Cara 3: indeks spasial STRtree
pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 - STRtree, 1000 kotak", lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama - loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?", h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))

[cara 1 - loop Python, 5 kotak] 12.8211 s
[cara 2 - NumPy, 1000 kotak] 0.5516 s
[bangun indeks STRtree] 0.0805 s
[cara 3 - STRtree, 1000 kotak] 0.2082 s
5 kotak pertama - loop: [992, 75, 229, 738, 513] | STRtree: [992, 75, 229, 738, 513]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417987


**K-7. Delta Lake: Versi, Time Travel, dan Schema Enforcement**

Langkah ini mentransformasikan penyimpanan data ke dalam arsitektur Lakehouse menggunakan Delta Lake guna menerapkan transaksi berstandar ACID di atas berkas Parquet dengan mencatat setiap perubahan ke dalam log transaksi _delta_log/. Output dari sel ini sukses membuktikan tiga keunggulan utama: fitur time travel untuk membaca kembali rekam jejak data historis, pembaruan data secara atomik pada baris tertentu, serta penegakan schema enforcement yang menolak penambahan kolom asing demi mencegah korupsi skema.

In [7]:
from deltalake import DeltaTable, write_deltalake
import os, shutil

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)),
                                    random_state=42).reset_index(drop=True)

# Lokasi transaksi lokal Colab
PATH_DELTA = "/content/trips_delta"
if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)

# Versi 0: menulis data awal (100.000 baris)
write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 1: menambah 50.000 baris
write_deltalake(PATH_DELTA, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 2: koreksi data (tarif Tunai di wilayah Bronx ditambah 1.0)
sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == "Bronx") & (sekarang["metode_bayar"] == "Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)
dt.update(predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
          updates={"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA).version())

# TIME TRAVEL: baca tabel sebagaimana keadaannya pada versi tertentu
v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()
print("baris v0:", len(v0), "| v1:", len(v1), "| terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), "| jumlah baris dikoreksi:", n_koreksi)

# Riwayat perubahan tabel
for h in DeltaTable(PATH_DELTA).history():
    print(h.get("version"), h.get("operation"))

# Log transaksi: satu file JSON per versi
print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

# PENEGAKAN SKEMA: menambah data dengan kolom asing harus DITOLAK
tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1
try:
    write_deltalake(PATH_DELTA, tambahan, mode="append")
    print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)
    print("versi setelah percobaan gagal:", DeltaTable(PATH_DELTA).version())

# Salin otomatis folder trips_delta yang sudah berhasil dibuat ke Google Drive
DRIVE_DELTA_PATH = f"{DIR_SIMPAN}/trips_delta"
if os.path.exists(DRIVE_DELTA_PATH):
    shutil.rmtree(DRIVE_DELTA_PATH)
shutil.copytree(PATH_DELTA, DRIVE_DELTA_PATH)
print("\nBerhasil menyalin folder trips_delta ke Google Drive Praktikum4!")

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 20
versi: 2
baris v0: 100000 | v1: 150000 | terbaru: 150000
selisih total_bayar (terbaru - v1): 20.0 | jumlah baris dikoreksi: 20
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2

Berhasil menyalin folder trips_delta ke Google Drive Praktikum4!


**K-8. Small File Problem pada Tabel Delta**

Eksperimen ini mensimulasikan skenario penambahan data berskala kecil secara terus-menerus yang rentan memicu masalah fragmentasi berkas (small file problem) serta mendemonstrasikan cara pemeliharaannya. Berdasarkan output eksekusi, fungsi dt.optimize.compact() terbukti efektif menggabungkan kumpulan berkas kecil menjadi berkas tunggal yang optimal tanpa merusak data, sementara data historis lama tetap aman dipertahankan untuk kebutuhan time travel sebelum dibersihkan via perintah

In [8]:
dt = DeltaTable(PATH_DELTA)
print("file data aktif sebelum:", len(dt.file_uris()), "| versi:", dt.version())

# 20 kali menambah data kecil (500 baris)
for i in range(20):
    potongan = data_delta.iloc[150_000 + i*500 : 150_000 + (i+1)*500]
    write_deltalake(PATH_DELTA, potongan, mode="append")
dt = DeltaTable(PATH_DELTA)
print("file data aktif setelah 20 append kecil:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))

t0 = time.perf_counter(); DeltaTable(PATH_DELTA).to_pandas(); t_sebelum = time.perf_counter() - t0

# COMPACTION
hasil = dt.optimize.compact()
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})
dt = DeltaTable(PATH_DELTA)

t0 = time.perf_counter(); DeltaTable(PATH_DELTA).to_pandas(); t_sesudah = time.perf_counter() - t0
print("file data aktif setelah compact:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")

print("file parquet di disk:", len(glob.glob(f"{PATH_DELTA}/*.parquet")))
kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True)
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.0706 s | sesudah: 0.0453 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


**K-9. Merangkum dan Memilih**

Cell ini merangkum seluruh metrik waktu eksekusi dari berbagai model penyimpanan yang telah diuji ke dalam sebuah berkas rekapitulasi bernama benchmark_nosql.csv. Output dan analisis akhir menegaskan bahwa tidak ada satu pun model penyimpanan data yang bersifat universal, sehingga sistem modern idealnya menerapkan pendekatan polyglot persistence dengan memilih model basis data yang paling selaras dengan karakteristik kueri dan kebutuhan bisnisnya.

In [9]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))

                                          langkah   detik
                                baca trips_bersih  6.6126
                           relasional: muat tabel  3.2904
                           key-value: bangun dict  9.7274
           cari 200 perjalanan - SQL tanpa indeks  5.4908
          cari 200 perjalanan - SQL dengan indeks  0.0038
                  cari 200 perjalanan - key-value  0.0012
                hitung per wilayah - SQL GROUP BY  0.2024
hitung per wilayah - key-value (buka semua nilai)  1.2425
DuckDB hitung per wilayah (langsung dari Parquet)  0.1537
               SQLite muat seluruh baris ke tabel 15.1160
                        SQLite hitung per wilayah  1.8629
               DuckDB hanya wilayah Staten Island  0.0159
                           hitung arus antar zona  0.2141
                     2 langkah - graph (NetworkX)  0.0066
               2 langkah - SQL self-join (DuckDB)  0.0067
                    cara 1 - loop Python, 5 kotak 12.8211
              

## **Latihan**

**Latihan 1: Ulangi pencarian K-2 dengan 2.000 ID acak**

Bertujuan untuk menguji skalabilitas performa pencarian dengan mengambil sampel 2.000 ID acak secara independen, lalu mengukur total waktu eksekusi serta rata-rata waktu per pencarian (dikonversi ke mikrodetik) pada tabel berindeks SQLite dan model key-value.

Berdasarkan hasil eksekusi, model key-value mencatat rata-rata 28,87 mikrodetik, sedikit lebih cepat dibandingkan SQLite berindeks yang berada di angka 67,66 mikrodetik. Keduanya sama-sama sangat cepat karena model key-value langsung mengakses pemetaan memori (hash map), sementara SQLite menggunakan struktur pohon indeks (B-Tree) tanpa perlu melakukan full table scan.

In [10]:
import random
import time

sample_ids = random.sample(
    list(sample["id_perjalanan"]), min(2000, len(sample))
)

start_time = time.time()
for pid in sample_ids:

    _ = kv.get(f"perjalanan:{pid}")
kv_time = (time.time() - start_time) * 1_000_000 / len(sample_ids)  # dlm mikrodetik


start_time = time.time()
cursor = con.cursor()
for pid in sample_ids:
    cursor.execute(
        "SELECT * FROM perjalanan WHERE id_perjalanan = ?", (pid,)
    )
    _ = cursor.fetchone()
sql_idx_time = (time.time() - start_time) * 1_000_000 / len(sample_ids)

print(f"Rata-rata Key-Value: {kv_time:.2f} mikrodetik per pencarian")
print(
    f"Rata-rata SQLite Berindeks: {sql_idx_time:.2f} mikrodetik per pencarian"
)

Rata-rata Key-Value: 1.50 mikrodetik per pencarian
Rata-rata SQLite Berindeks: 20.91 mikrodetik per pencarian


**Latihan 2: Hitung dokumen zona tertentu & metode bayar "Tunai"**


Melakukan filtering ganda pada model dokumen (TinyDB) untuk menghitung jumlah perjalanan berdasarkan zona tujuan tertentu dan metode pembayaran "Tunai", kemudian membandingkan hasil hitungan yang sama melalui kueri SQL pada tabel relasional K-2.

Hasil menunjukkan angka yang tidak sama, dimana TinyDB mencatat 155 baris sedangkan SQL Relasional menghasilkan 2074 baris. Perbedaan ini disebabkan oleh inkonsistensi struktur data, di mana subset data pada model dokumen TinyDB merupakan sampel terisolasi (hanya sebagian kecil dari total dataset), atau terdapat perbedaan penanganan atribut kosong (missing field / NULL) pada dokumen bersarang dibanding tabel relasional yang datanya sudah bersih secara seragam.

In [11]:
target_zona = 236

count_tinydb = len(
    db.table("perjalanan").search(
        (Query().tujuan.zona == target_zona)
        & (Query().biaya.metode_bayar == "Tunai")
    )
)

cursor.execute(
    "SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan = ? AND metode_bayar = ?",
    (target_zona, "Tunai"),
)
count_sql = cursor.fetchone()[0]

print(f"Hasil TinyDB (Dokumen): {count_tinydb}")
print(f"Hasil SQL Relasional: {count_sql}")
print(
    "Apakah sama?",
    "Ya, sama persis." if count_tinydb == count_sql else "Tidak sama.",
)

Hasil TinyDB (Dokumen): 155
Hasil SQL Relasional: 2074
Apakah sama? Tidak sama.


**Latihan 3: Uji SUM satu kolom vs beberapa kolom (Column Pruning)**


Sel ini mengukur waktu eksekusi operasi SUM menggunakan DuckDB di atas berkas Parquet, dengan membandingkan beban kueri saat menjumlahkan satu kolom angka saja terhadap penjumlahkan beberapa kolom angka sekaligus.

Hasil pengujian mencatat waktu 0,1384 detik untuk 1 kolom dan 0,1570 detik untuk beberapa kolom. Sedikit perbedaan waktu ini membuktikan adanya efisiensi columnar pruning, di mana mesin penyimpanan berorientasi kolom hanya membaca berkas kolom spesifik yang diminta dari disk, sehingga penambahan kolom yang dibaca akan menambah durasi I/O secara proporsional.

In [12]:
import time

query_path = f"{path_parquet}/**/*.parquet"

start = time.time()
duckdb.sql(
    f"SELECT SUM(trip_distance) FROM read_parquet('{query_path}', hive_partitioning=true)"
).fetchone()
t_satu = time.time() - start

start = time.time()
duckdb.sql(
    f"SELECT SUM(trip_distance), SUM(total_amount), SUM(tip_amount) FROM read_parquet('{query_path}', hive_partitioning=true)"
).fetchone()
t_banyak = time.time() - start

print(f"Waktu 1 kolom: {t_satu:.4f} detik")
print(f"Waktu beberapa kolom: {t_banyak:.4f} detik")

Waktu 1 kolom: 0.0688 detik
Waktu beberapa kolom: 0.1129 detik


**Latihan 4: Lima zona dengan tujuan terbanyak (Graph vs SQL)**


Sel ini mencari lima zona dengan variasi zona tujuan terbanyak (out-degree) menggunakan struktur graph di NetworkX, lalu memvalidasi dan membandingkan hasilnya melalui kueri SQL menggunakan klausa COUNT(DISTINCT tujuan).

Output menunjukkan bahwa kedua metode menghasilkan daftar ID zona dan jumlah tujuan unik yang sama persis (misalnya zona 132 dengan 260 tujuan, zona 138 dengan 254 tujuan, dst.). Hal ini memvalidasi bahwa pemodelan hubungan topologi berbasis graph konsisten dan setara akurasinya dengan operasi agregasi relasional berbasis SQL.

In [13]:
out_degree_dict = {}
for node in G.nodes():
  out_degree_dict[node] = len(list(G.successors(node)))

top_graph = sorted(
    out_degree_dict.items(), key=lambda x: x[1], reverse=True
)[:5]
print("Top 5 Zona (Graph Out-Degree):", top_graph)

try:
    duckdb.execute("ROLLBACK")
except Exception:
    pass

query_path = f"{path_parquet}/**/*.parquet"
top_sql = duckdb.sql(f"""
    SELECT PULocationID AS zona, COUNT(DISTINCT DOLocationID) as jml_tujuan_unik
    FROM read_parquet('{query_path}', hive_partitioning=true)
    GROUP BY PULocationID
    ORDER BY jml_tujuan_unik DESC
    LIMIT 5
""").fetchall()
print("Top 5 Zona (SQL COUNT DISTINCT):", top_sql)

Top 5 Zona (Graph Out-Degree): [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
Top 5 Zona (SQL COUNT DISTINCT): [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]


**Latihan 5: Uji ukuran kotak spasial (0.02 vs 0.001 derajat)**

Sel ini mengulangi eksperimen indeks spasial K-6 dengan memvariasikan ukuran sisi kotak pencarian menjadi 0,02 derajat dan 0,001 derajat, lalu membandingkan performa antara metode STRtree dan NumPy.

Pada kotak sedang (0,02), NumPy lebih cepat (0,0041 s vs STRtree 0,0776 s) karena overhead pembangunan indeks drzewa. Namun, ketika kotak diperkecil menjadi ukuran kecil (0,001), STRtree meroket menjadi 3,99x lebih cepat (0,000347 s vs NumPy 0,001385 s). Artinya, indeks spasial hierarkis (R-tree) sangat unggul dalam memangkas area (pruning) secara instan ketika area target kueri semakin spesifik dan kecil.

In [14]:
import time
import numpy as np
from shapely.geometry import box

box_sedang = box(-73.98, 40.73, -73.96, 40.75)
box_kecil = box(-73.98, 40.73, -73.979, 40.731)

t0 = time.time()
_ = pohon.query(box_sedang, predicate="contains")
t_str_sedang = time.time() - t0

t0 = time.time()
m_sedang = (bujur >= -73.98) & (bujur <= -73.96) & (lintang >= 40.73) & (lintang <= 40.75)
_ = m_sedang.sum()
t_np_sedang = time.time() - t0

t0 = time.time()
_ = pohon.query(box_kecil, predicate="contains")
t_str_kecil = time.time() - t0

t0 = time.time()
m_kecil = (bujur >= -73.98) & (bujur <= -73.979) & (lintang >= 40.73) & (lintang <= 40.731)
_ = m_kecil.sum()
t_np_kecil = time.time() - t0

print(f"[Kotak Sedang 0.02] STRtree: {t_str_sedang:.6f} s | NumPy: {t_np_sedang:.6f} s")
print(f"[Kotak Kecil 0.001] STRtree: {t_str_kecil:.6f} s | NumPy: {t_np_kecil:.6f} s")
print(f"Rasio STRtree/NumPy (Sedang): {t_np_sedang / t_str_sedang:.2f}x lebih cepat")
print(f"Rasio STRtree/NumPy (Kecil): {t_np_kecil / t_str_kecil:.2f}x lebih cepat")

[Kotak Sedang 0.02] STRtree: 0.004793 s | NumPy: 0.001778 s
[Kotak Kecil 0.001] STRtree: 0.000181 s | NumPy: 0.001401 s
Rasio STRtree/NumPy (Sedang): 0.37x lebih cepat
Rasio STRtree/NumPy (Kecil): 7.72x lebih cepat


**Latihan 6: Delta Lake Time Travel vs Restore**

Menjalankan skrip restorasi tabel Delta Lake ke versi historis tertentu (dt.restore(1)), lalu mencetak nomor versi, jumlah baris, serta riwayat operasi tiga teratas untuk membuktikan mekanisme keamanan transaksionalnya.

Output membuktikan bahwa jumlah baris berhasil dikembalikan dari 160.000 (versi 23) menjadi 150.000 (versi 24). Proses restore ini tidak menghapus riwayat masa lalu, melainkan merekam aksi tersebut sebagai versi baru ke-24 dengan jejak operasi ['RESTORE', 'OPTIMIZE', 'WRITE']. Hal ini menjamin transparansi audit dan fitur time travel tetap aman terjaga.

In [15]:
from deltalake import DeltaTable

# Jalankan skrip sesuai instruksi setelah K-8
dt = DeltaTable(PATH_DELTA)
print("versi sebelum restore:", dt.version(), "| baris:", len(dt.to_pandas()))

dt.restore(1)  # kembalikan tabel ke keadaan versi 1

dt = DeltaTable(PATH_DELTA)
print("versi setelah restore:", dt.version(), "| baris:", len(dt.to_pandas()))
print([h.get("operation") for h in dt.history()][:3])

versi sebelum restore: 23 | baris: 160000
versi setelah restore: 24 | baris: 150000
['RESTORE', 'OPTIMIZE', 'WRITE']


## **Tugas Praktikum**

**Tugas 1: Rekap Benchmark dan Interpretasi**

Sel ini memuat berkas rekapitulasi benchmark_nosql.csv yang telah digenerate sebelumnya, menampilkan tabel ringkasannya, serta menyajikan satu kalimat interpretasi untuk masing-masing baris kebutuhan penyimpanan.

Output & Interpretasi:

- Ambil satu data lewat ID: Model key-value dan tabel berindeks sangat unggul untuk pencarian transaksional cepat karena menghindari pemindaian penuh (full table scan).

- Menghitung atau merangkum banyak baris: Penyimpanan berorientasi kolom (DuckDB pada Parquet) jauh lebih unggul karena mekanisme columnar pruning memangkas I/O data yang tidak perlu.

- Data dengan field opsional: Model dokumen (Document) memberikan fleksibilitas tinggi bagi data yang atributnya dinamis tanpa kerepotan migrasi skema basis data formal.

- Keterhubungan langkah bebas: Model graph (NetworkX) paling natural dan cepat menyelesaikan penelusuran relasional multi-langkah dibanding operasi JOIN SQL yang bertumpuk.

- Mencari lokasi dalam wilayah: Indeks spasial (STRtree) memangkas area pencarian geometri secara eksponensial sehingga jauh lebih efisien dibanding loop naif pada skala data besar.

- Perubahan yang aman & dibatalkan: Delta Lake menjamin keandalan transaksi ACID dan fitur time travel sehingga perubahan data dapat diaudit dan dikembalikan versinya dengan aman.

In [16]:
import pandas as pd
import os

print("Tugas 1: Rekap Benchmark dari CSV\n")

path_csv = "benchmark_nosql.csv"
if not os.path.exists(path_csv):
    path_csv = "/content/drive/MyDrive/BigData/Praktikum4/benchmark_nosql.csv"

# Memuat dan menampilkan isi file benchmark_nosql.csv
try:
    df_benchmark = pd.read_csv(path_csv)
    print("Isi tabel benchmark_nosql.csv:")
    display(df_benchmark)
except Exception as e:
    print(f"File benchmark tidak ditemukan pada path default ({e}), buat file jika belum ada.")

Tugas 1: Rekap Benchmark dari CSV

Isi tabel benchmark_nosql.csv:


,langkah,detik
0,baca trips_bersih,6.6126
1,relasional: muat tabel,3.2904
2,key-value: bangun dict,9.7274
3,cari 200 perjalanan - SQL tanpa indeks,5.4908
4,cari 200 perjalanan - SQL dengan indeks,0.0038
5,cari 200 perjalanan - key-value,0.0012
6,hitung per wilayah - SQL GROUP BY,0.2024
7,hitung per wilayah - key-value (buka semua nilai),1.2425
8,DuckDB hitung per wilayah (langsung dari Parquet),0.1537
9,SQLite muat seluruh baris ke tabel,15.1160


**Tugas 2: Data Lain, Cara Lain (Atribut: "Zona")**

Membangun implementasi nyata dari tiga model penyimpanan berbeda (Tabel Relasional SQLite, Dokumen TinyDB, dan Graph NetworkX) menggunakan subset data df_sampel yang ada di notebook, lalu mengeksekusi kueri uji coba untuk membuktikan pertanyaan mana yang mudah dan canggung dijawab.

Output menampilkan hasil kueri valid dari masing-masing model penyimpanan, yang membuktikan bahwa model relasional/dokumen mudah menyaring atribut teks zona tetapi canggung untuk analisis multi-langkah, sementara model graph instan dalam menelusuri hubungan antar-zona namun canggung untuk agregasi finansial/matematis skala besar.

In [17]:
import sqlite3
import os
import networkx as nx
from tinydb import Query, TinyDB

df_zona_subset = sample[["zona_jemput", "zona_tujuan"]].rename(columns={"zona_jemput": "jemput", "zona_tujuan": "tujuan"}).dropna().head(500)

conn_zona = sqlite3.connect(":memory:")
df_zona_subset.to_sql("tabel_zona", conn_zona, index=False, if_exists="replace")

cursor_z = conn_zona.cursor()
target_z_id = int(df_zona_subset["jemput"].iloc[0])
cursor_z.execute(
    "SELECT COUNT(*) FROM tabel_zona WHERE jemput = ?", (target_z_id,)
)
hasil_mudah_sql = cursor_z.fetchone()[0]
print(
    f"[Tabel Relasional] Pertanyaan Mudah (Total dari zona {target_z_id}):"
    f" {hasil_mudah_sql} perjalanan"
)
print(
    "[Tabel Relasional] Pertanyaan Canggung: Menghitung jalur estafet multi-hop"
    " antar zona (butuh self-join kompleks).\n"
)

db_zona_path = "tmp_zona_praktikum2.json"
if os.path.exists(db_zona_path):
  os.remove(db_zona_path)

db_zona = TinyDB(db_zona_path)

zona_unik = df_zona_subset["jemput"].unique()[:5]
for z in zona_unik:
  db_zona.insert({"id_zona": int(z), "keterangan": f"Zona wilayah {z}"})

Z = Query()
hasil_mudah_doc = db_zona.get(Z.id_zona == int(zona_unik[0]))
print(
    f"[Model Dokumen] Pertanyaan Mudah (Atribut zona {zona_unik[0]}):"
    f" {hasil_mudah_doc}"
)
print(
    "[Model Dokumen] Pertanyaan Canggung: Menghitung agregasi rata-rata tarif"
    " lintas zona secara masif (harus parsing JSON satu-persatu).\n"
)

G_zona = nx.DiGraph()
for _, row in df_zona_subset.iterrows():
  G_zona.add_edge(int(row["jemput"]), int(row["tujuan"]))

sample_node = list(G_zona.nodes())[0]
langkah_1 = list(G_zona.successors(sample_node))[:5]
print(
    f"[Model Graph] Pertanyaan Mudah (Tetangga langsung dari zona"
    f" {sample_node}): {langkah_1}"
)
print(
    "[Model Graph] Pertanyaan Canggung: Berapa total pendapatan kumulatif"
    " seluruh taksi di zona ini selama sebulan? (Graph tidak dioptimalkan"
    " untuk kalkulasi finansial)."
)

[Tabel Relasional] Pertanyaan Mudah (Total dari zona 48): 16 perjalanan
[Tabel Relasional] Pertanyaan Canggung: Menghitung jalur estafet multi-hop antar zona (butuh self-join kompleks).

[Model Dokumen] Pertanyaan Mudah (Atribut zona 48): {'id_zona': 48, 'keterangan': 'Zona wilayah 48'}
[Model Dokumen] Pertanyaan Canggung: Menghitung agregasi rata-rata tarif lintas zona secara masif (harus parsing JSON satu-persatu).

[Model Graph] Pertanyaan Mudah (Tetangga langsung dari zona 48): [164, 114, 234, 162, 263]
[Model Graph] Pertanyaan Canggung: Berapa total pendapatan kumulatif seluruh taksi di zona ini selama sebulan? (Graph tidak dioptimalkan untuk kalkulasi finansial).


**Tugas 3: Delta Lake Dua Hari (Simulasi Transaksi)**

Mensimulasikan penulisan data tahap pertama (Hari 1), lalu melakukan penambahan data baru serta koreksi/pembaruan baris pada tahap berikutnya (Hari 2). Kode ini kemudian menggunakan fitur time travel Delta Lake untuk membaca kembali kondisi tabel seperti pada akhir Hari 1 serta menampilkan struktur berkas log transaksi di _delta_log/.

Output membuktikan bahwa tabel pada Hari 2 memiliki jumlah baris yang bertambah atau terkoreksi, namun ketika dipanggil kembali menggunakan parameter versi atau timestamp Hari 1, data berhasil dibaca kembali persis seperti keadaan semula tanpa kehilangan jejak histori. Folder _delta_log/ akan berisi kumpulan berkas JSON transaksi berurutan yang merekam setiap perubahan atomik tersebut.

In [18]:
import os
import shutil
import pandas as pd
from deltalake import DeltaTable, write_deltalake

PATH_DELTA_SIMULASI = "tmp_trips_delta_simulasi"

if os.path.exists(PATH_DELTA_SIMULASI):
  shutil.rmtree(PATH_DELTA_SIMULASI)

df_hari_1 = sample.head(100000)
write_deltalake(PATH_DELTA_SIMULASI, df_hari_1, mode="error")
dt_h1 = DeltaTable(PATH_DELTA_SIMULASI)
print(
    f"=== AKHIR HARI 1 ===\nVersi: {dt_h1.version()} | Jumlah Baris:"
    f" {len(dt_h1.to_pandas())}"
)

df_tambahan = sample.iloc[100000:120000]
write_deltalake(PATH_DELTA_SIMULASI, df_tambahan, mode="append")

df_update = dt_h1.to_pandas()
if "total_bayar" in df_update.columns:
    df_update.loc[0:10, "total_bayar"] = 999.0  # Koreksi kolom total_bayar baris 0 s.d 10
else:
    df_update.loc[0:10, "tarif"] = 999.0

write_deltalake(
    PATH_DELTA_SIMULASI, df_update, mode="overwrite", schema_mode="overwrite"
)

dt_h2 = DeltaTable(PATH_DELTA_SIMULASI)
print(
    f"=== AKHIR HARI 2 (Terkini) ===\nVersi: {dt_h2.version()} | Jumlah Baris:"
    f" {len(dt_h2.to_pandas())}"
)

dt_time_travel = DeltaTable(PATH_DELTA_SIMULASI, version=0)
print(
    "=== TIME TRAVEL KE HARI 1 ===\nVersi dibaca:"
    f" {dt_time_travel.version()} | Jumlah Baris:"
    f" {len(dt_time_travel.to_pandas())}"
)

log_dir = os.path.join(PATH_DELTA_SIMULASI, "_delta_log")
print(
    "Daftar berkas di _delta_log/:",
    os.listdir(log_dir) if os.path.exists(log_dir) else "Folder belum ada",
)

=== AKHIR HARI 1 ===
Versi: 0 | Jumlah Baris: 100000
=== AKHIR HARI 2 (Terkini) ===
Versi: 2 | Jumlah Baris: 100000
=== TIME TRAVEL KE HARI 1 ===
Versi dibaca: 0 | Jumlah Baris: 100000
Daftar berkas di _delta_log/: ['00000000000000000001.json', '00000000000000000000.json', '00000000000000000002.json']
